In [37]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingClassifier 

In [ ]:
# Calculate Average Price Feature from one_hot_mama.csv
import pandas as pd
import numpy as np

# Read the year-specific data
#one_hot_data = pd.read_csv('data/one_hot_mama.csv')

# # Find all price columns (YEAR_price_usd_tonne)
# price_cols = [col for col in one_hot_data.columns if col.endswith('_price_usd_tonne')]
# print(f"Found {len(price_cols)} price columns: {sorted(price_cols)}")

# # Calculate average price: sum of non-zero values divided by count of non-zero entries
# def calculate_avg_price(row):
#     """Calculate average price from year-specific price columns, excluding zeros and NaNs"""
#     prices = row[price_cols].replace(0, np.nan)  # Treat 0 as missing
#     non_zero_prices = prices.dropna()
    
#     if len(non_zero_prices) == 0:
#         return np.nan
#     else:
#         return non_zero_prices.sum() / len(non_zero_prices)

# # Apply to each row
# avg_price_feature = one_hot_data.apply(calculate_avg_price, axis=1)

# # Read the average_data file and add the new feature
# average_data = pd.read_csv('data/average_data.csv')
# average_data['avg_price_usd'] = avg_price_feature

# # Save the updated file
# average_data.to_csv('data/average_data.csv', index=False)

# print(f"\nAdded 'avg_price_usd' feature to average_data.csv")
# print(f"New feature statistics:")
# print(average_data['avg_price_usd'].describe())
# print(f"\nNull values: {average_data['avg_price_usd'].isna().sum()}")


Found 21 price columns: ['2000_price_usd_tonne', '2001_price_usd_tonne', '2002_price_usd_tonne', '2003_price_usd_tonne', '2004_price_usd_tonne', '2005_price_usd_tonne', '2006_price_usd_tonne', '2007_price_usd_tonne', '2008_price_usd_tonne', '2009_price_usd_tonne', '2010_price_usd_tonne', '2011_price_usd_tonne', '2012_price_usd_tonne', '2013_price_usd_tonne', '2014_price_usd_tonne', '2015_price_usd_tonne', '2016_price_usd_tonne', '2017_price_usd_tonne', '2018_price_usd_tonne', '2019_price_usd_tonne', '2020_price_usd_tonne']

Added 'avg_price_usd' feature to average_data.csv
New feature statistics:
count      352.000000
mean      1791.852012
std       2915.813909
min         13.687000
25%         76.460536
50%        101.400000
75%       1894.250571
max      19845.493636
Name: avg_price_usd, dtype: float64

Null values: 22


In [8]:
import pandas as pd
import re
from itertools import combinations

In [ ]:
mother_file = pd.read_csv('data/average_data.csv') #average


{'total_reserves_minerals': ['total_reserves_minerals'], 'total_reserves_commodities': ['total_reserves_commodities'], 'avg_reserves_grade_ppm': ['avg_reserves_grade_ppm'], 'avg_commodity_recovery_rate': ['avg_commodity_recovery_rate'], 'avg_commodities_grade_ppm': ['avg_commodities_grade_ppm'], 'total_coal': ['total_coal'], 'total_commodities': ['total_commodities'], 'total_minerals': ['total_minerals'], 'mining_salary': ['mining_salary'], 'still_operating': ['still_operating'], 'country': ['country_Argentina', 'country_Australia', 'country_Bolivia', 'country_Bosnia and Herzegovina', 'country_Brazil', 'country_Canada', 'country_Chile', 'country_China', 'country_Colombia', "country_Cote d'Ivoire", 'country_Cuba', 'country_DR Congo', 'country_Dominican Republic', 'country_Finland', 'country_Ghana', 'country_Guatemala', 'country_Guinea', 'country_India', 'country_Indonesia', 'country_Jamaica', 'country_Kazakhstan', 'country_Kyrgyzstan', 'country_Liberia', 'country_Madagascar', 'country_M

In [20]:
# Group average_data features and exclude target/data identifiers
import re

average_data = pd.read_csv('data/average_data.csv')

YEAR_PREFIX_RE = re.compile(r'^(\d{4})_(.+)$')
ignore_cols = {'facility_id', 'LOM', 'LOM_in_Buckets', 'LOM_Buckets', 'LOM_quintile'}
feature_groups = {}

for col in average_data.columns:
    if col in ignore_cols:
        continue

    if col.startswith('country_'):
        group_name = 'country'
    elif col.startswith('primary_commodity_'):
        group_name = 'primary_commodity'
    else:
        match = YEAR_PREFIX_RE.match(col)
        group_name = match.group(2) if match else col

    feature_groups.setdefault(group_name, []).append(col)

feature_group_names = sorted(feature_groups)
print(f"Created {len(feature_group_names)} feature groups from average_data.csv")
print('Example groups:')
for group_name in ('country', 'primary_commodity'):
    if group_name in feature_groups:
        print(f"  {group_name}: {len(feature_groups[group_name])} columns")

print(f"\nAll feature groups: {feature_group_names}")


Created 14 feature groups from average_data.csv
Example groups:
  country: 47 columns
  primary_commodity: 11 columns

All feature groups: ['avg_commodities_grade_ppm', 'avg_commodity_recovery_rate', 'avg_price_usd', 'avg_reserves_grade_ppm', 'commodity_var', 'country', 'mining_salary', 'primary_commodity', 'still_operating', 'total_coal', 'total_commodities', 'total_minerals', 'total_reserves_commodities', 'total_reserves_minerals']


# Regressor

### Average

In [27]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import heapq
from itertools import count
import math

# Load the average_data file and target variable
average_data = pd.read_csv('data/average_data.csv')
target = 'LOM_in_Buckets'

search_groups = sorted(feature_groups)
max_results = 20
best_results = []
tie_breaker = count()

total_combos = sum(math.comb(len(search_groups), r) for r in range(1, len(search_groups) + 1))

with tqdm(total=total_combos, desc='Training combos') as pbar:
    for r in range(1, len(search_groups) + 1):
        for combo in combinations(search_groups, r):
            selected_cols = [col for group in combo for col in feature_groups[group]]
            X = average_data[selected_cols].select_dtypes(include='number').copy()
            y = average_data[target].copy()

            # Replace infinite values and drop rows with missing values
            X = X.replace([np.inf, -np.inf], np.nan)
            valid_mask = X.notna().all(axis=1) & y.notna()
            X = X.loc[valid_mask]
            y = y.loc[valid_mask]

            if X.shape[0] < 10 or X.shape[1] == 0:
                pbar.update(1)
                continue

            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            model = RandomForestRegressor(random_state=42, n_estimators=100)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            score = r2_score(y_test, y_pred)

            record = {
                'feature_groups': combo,
                'num_groups': len(combo),
                'num_columns': len(selected_cols),
                'num_rows': X.shape[0],
                'r2_test': score,
            }

            heap_item = (score, next(tie_breaker), record)
            if len(best_results) < max_results:
                heapq.heappush(best_results, heap_item)
            else:
                heapq.heappushpop(best_results, heap_item)

            pbar.update(1)

best_sorted = [record for _, _, record in sorted(best_results, key=lambda x: x[0], reverse=True)]
combo_df = pd.DataFrame(best_sorted)
print(combo_df.to_string(index=False))

combo_df.to_csv('data/feature_group_regression_results.csv', index=False)
print('Saved top 20 feature-group combination regression results to data/feature_group_regression_results.csv')


Training combos: 100%|██████████| 16383/16383 [15:49<00:00, 17.26it/s]

                                                                                                                                                        feature_groups  num_groups  num_columns  num_rows  r2_test
                 (avg_commodities_grade_ppm, avg_reserves_grade_ppm, commodity_var, country, mining_salary, primary_commodity, total_coal, total_reserves_commodities)           8           64        72 0.679516
            (avg_commodities_grade_ppm, avg_reserves_grade_ppm, commodity_var, country, mining_salary, primary_commodity, still_operating, total_reserves_commodities)           8           64        72 0.679516
                             (avg_commodities_grade_ppm, avg_reserves_grade_ppm, commodity_var, country, mining_salary, primary_commodity, total_reserves_commodities)           7           63        72 0.675087
                                    (avg_commodities_grade_ppm, avg_reserves_grade_ppm, commodity_var, country, mining_salary, total_coal, total_reserves_co

### Printing feature importance of best from regressor

In [36]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Separate model using the best feature set outside the loop
selected_features = [
    'avg_commodities_grade_ppm',
    'avg_reserves_grade_ppm',
    'commodity_var',
    'country',
    'mining_salary',
    'primary_commodity',
    'total_coal',
    'total_reserves_commodities',
]

average_data = pd.read_csv('data/average_data.csv')
target = 'LOM_in_Buckets'

missing_features = [f for f in selected_features if f not in average_data.columns]
if missing_features:
    raise KeyError(f"Missing selected features in average_data.csv: {missing_features}")

X = average_data[selected_features].copy()
y = average_data[target].copy()

valid_mask = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_mask]
y = y.loc[valid_mask]

# Encode categorical features
for cat_col in ['country', 'primary_commodity']:
    X[cat_col] = X[cat_col].astype('category').cat.codes

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42, n_estimators=100)
model.fit(X_train, y_train)

print(f"Separate selected-feature model R2: {r2_score(y_test, model.predict(X_test)):.4f}\n")

importances = (
    pd.Series(model.feature_importances_, index=X_test.columns)
    .sort_values(ascending=False)
    .head(15)
)

print("Top 15 feature importances:")
for feat, imp in importances.items():
    bar = "█" * int(imp * 300)
    print(f"  {feat:<45} {imp:.4f}  {bar}")


Separate selected-feature model R2: 0.5709

Top 15 feature importances:
  total_reserves_commodities                    0.4009  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  commodity_var                                 0.1706  ███████████████████████████████████████████████████
  primary_commodity                             0.1625  ████████████████████████████████████████████████
  country                                       0.1112  █████████████████████████████████
  avg_reserves_grade_ppm                        0.0700  ████████████████████
  avg_commodities_grade_ppm                     0.0582  █████████████████
  mining_salary                                 0.0266  ███████
  total_coal                                    0.0000  


# Classifier

### add quintiles to average data

In [44]:
# info abt LoM distribution
mother_file = pd.read_csv('data/average_data.csv')
#print(mother_file["LOM_in_Buckets"].describe().round(2).to_string())
 
 # quintine distribution
quintile_probs = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
quintile_edges = mother_file["LOM_in_Buckets"].quantile(quintile_probs).tolist()
 
quintile_edges = sorted(set(quintile_edges))
 
#print("Quintile cut-points (0%, 20%, 40%, 60%, 80%, 100%):")
for p, v in zip(quintile_probs, mother_file["LOM_in_Buckets"].quantile(quintile_probs)):
    print(f"  {int(p*100):>3}th percentile - {v:.1f} years")
# putting them in bins 
mother_file["LOM_quintile"] = pd.qcut(
    mother_file["LOM_in_Buckets"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    duplicates="drop",
)
 
# print("Records per quintile bin:")
# print(mother_file["LOM_quintile"].value_counts().sort_index().to_string())
 
# # Bin ranges for reference
# print("Bin value ranges:")
for bin_name in mother_file["LOM_quintile"].cat.categories:
    subset = mother_file.loc[mother_file["LOM_quintile"] == bin_name, "LOM_in_Buckets"]
    print(f"  {bin_name:<18} → [{subset.min():.0f}, {subset.max():.0f}] yrs  (n={len(subset)})")

mother_file.to_csv('data/average_data.csv', index=False)
 

    0th percentile - 1.0 years
   20th percentile - 9.0 years
   40th percentile - 14.0 years
   60th percentile - 29.0 years
   80th percentile - 50.4 years
  100th percentile - 467.0 years
  Q1                 → [1, 9] yrs  (n=77)
  Q2                 → [10, 14] yrs  (n=76)
  Q3                 → [15, 29] yrs  (n=74)
  Q4                 → [30, 50] yrs  (n=72)
  Q5                 → [51, 467] yrs  (n=75)


### Average Data

In [52]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import heapq
from itertools import count
import math

# Load the average_data file and target variable
average_data = pd.read_csv('data/average_data.csv')
target = 'LOM_quintile'
average_data = average_data.dropna(subset=['LOM_in_Buckets'])
search_groups = sorted(feature_groups)
max_results = 20
best_results = []
tie_breaker = count()

total_combos = sum(math.comb(len(search_groups), r) for r in range(1, len(search_groups) + 1))

with tqdm(total=total_combos, desc='Classifier combos') as pbar:
    for r in range(1, len(search_groups) + 1):
        for combo in combinations(search_groups, r):
            selected_cols = [col for group in combo for col in feature_groups[group]]
            X = average_data[selected_cols].select_dtypes(include='number').copy()
            y = average_data[target].copy()

            X = X.replace([np.inf, -np.inf], np.nan)
            valid_mask = X.notna().all(axis=1) & y.notna()
            X = X.loc[valid_mask]
            y = y.loc[valid_mask]

            if X.shape[0] < 10 or X.shape[1] == 0:
                pbar.update(1)
                continue

            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            model = RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=2,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1,
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred)
            balanced_acc = balanced_accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
            f1_w = report['weighted avg']['f1-score']

            record = {
                'feature_groups': combo,
                'num_columns': len(selected_cols),
                'accuracy': accuracy,
                'balanced_accuracy': balanced_acc,
                'f1_weighted': f1_w,
                'precision_weighted': report['weighted avg']['precision'],
                'recall_weighted': report['weighted avg']['recall'],
            }
            heap_item = (f1_w, next(tie_breaker), record)
            if len(best_results) < max_results:
                heapq.heappush(best_results, heap_item)
            else:
                heapq.heappushpop(best_results, heap_item)

            pbar.update(1)

best_sorted = [record for _, _, record in sorted(best_results, key=lambda x: x[0], reverse=True)]
combo_df = pd.DataFrame(best_sorted)
print(combo_df.to_string(index=False))

combo_df.to_csv('data/feature_group_classifier_results.csv', index=False)
print('Saved top 20 feature-group classifier results to data/feature_group_classifier_results.csv')


Classifier combos:   0%|          | 3/16383 [00:00<49:57,  5.46it/s]  /Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project/.venv1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Classifier combos:   0%|          | 4/16383 [00:00<45:14,  6.03it/s]/Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project/.venv1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Classifier combos:   0%|          | 28/16383 [00:04<41:15,  6.61it/s]/Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project/.venv1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Classifier combos:   0%|          | 29/16383 [

                                                                                                                                                                                                                                                    feature_groups  num_columns  accuracy  balanced_accuracy  f1_weighted  precision_weighted  recall_weighted
                                                                                                           (avg_commodity_recovery_rate, avg_reserves_grade_ppm, country, still_operating, total_coal, total_commodities, total_minerals, total_reserves_minerals)           54  0.666667           0.600000     0.724537            0.861111         0.666667
                                                                                                                                                  (avg_commodity_recovery_rate, avg_reserves_grade_ppm, mining_salary, total_commodities, total_reserves_minerals)            5  0.666667           0.6000

### Year Based Data

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import heapq
from itertools import count
import math

# Load the one_hot_mama file and target variable
average_data = pd.read_csv('data/one_hot_mama.csv')
target = 'LOM_quintile'
average_data = average_data.dropna(subset=['LOM_in_Buckets'])
search_groups = sorted(feature_groups)
max_results = 20
best_results = []
tie_breaker = count()

total_combos = sum(math.comb(len(search_groups), r) for r in range(1, len(search_groups) + 1))

with tqdm(total=total_combos, desc='Classifier combos') as pbar:
    for r in range(1, len(search_groups) + 1):
        for combo in combinations(search_groups, r):
            selected_cols = [col for group in combo for col in feature_groups[group]]
            X = average_data[selected_cols].select_dtypes(include='number').copy()
            y = average_data[target].copy()

            X = X.replace([np.inf, -np.inf], np.nan)
            valid_mask = X.notna().all(axis=1) & y.notna()
            X = X.loc[valid_mask]
            y = y.loc[valid_mask]

            if X.shape[0] < 10 or X.shape[1] == 0:
                pbar.update(1)
                continue

            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            model = RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=2,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1,
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred)
            balanced_acc = balanced_accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
            f1_w = report['weighted avg']['f1-score']

            record = {
                'feature_groups': combo,
                'num_columns': len(selected_cols),
                'accuracy': accuracy,
                'balanced_accuracy': balanced_acc,
                'f1_weighted': f1_w,
                'precision_weighted': report['weighted avg']['precision'],
                'recall_weighted': report['weighted avg']['recall'],
            }
            heap_item = (f1_w, next(tie_breaker), record)
            if len(best_results) < max_results:
                heapq.heappush(best_results, heap_item)
            else:
                heapq.heappushpop(best_results, heap_item)

            pbar.update(1)

best_sorted = [record for _, _, record in sorted(best_results, key=lambda x: x[0], reverse=True)]
combo_df = pd.DataFrame(best_sorted)
print(combo_df.to_string(index=False))

combo_df.to_csv('data/feature_group_classifier_results.csv', index=False)
print('Saved top 20 feature-group classifier results to data/feature_group_classifier_results.csv')
